In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import lxml

# Question 1: [IPO] Withdrawn IPOs by Company Type

In [2]:
url = "https://www.iposcoop.com/ipos-recently-filed/"
recent_ipo = pd.read_html(url)
df_base = recent_ipo[0]
df = df_base[df_base['Expected To Trade'] == 'Withdrawn']

In [3]:
df['Company Type'] = np.select(
    [
        df['Company'].str.contains('Technologies', na=False),
        df['Company'].str.contains('Acquisition Corp|Acquisition Corporation|Corp', na=False, regex=True),
        df['Company'].str.contains('Inc|Incorporated', na=False, regex=True),
        df['Company'].str.contains('Group', na=False, regex=True),
        df['Company'].str.contains('Ltd|Limited', na=False, regex=True),
        df['Company'].str.contains('Holdings|Holding', na=False, regex=True),
    ],
    ['Technologies', 'Acquisition Corp', 'Inc.', 'Group', 'Limited', 'Holdings'],
    default='Others',
)
df.info()

<class 'pandas.DataFrame'>
Index: 34 entries, 1 to 499
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   File Date             34 non-null     str    
 1   Company               34 non-null     str    
 2   Symbol                34 non-null     str    
 3   Managers              34 non-null     str    
 4   Shares (millions)     34 non-null     float64
 5   Price Low             28 non-null     str    
 6   Price High            28 non-null     str    
 7   Est $ Vol (millions)  34 non-null     str    
 8   Expected To Trade     34 non-null     str    
 9   SCOOP Rating          34 non-null     str    
 10  Company Type          34 non-null     str    
dtypes: float64(1), str(10)
memory usage: 3.2 KB


In [4]:
df_clean = df.copy()
df_clean.head()

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type
1,2026-09-17,C2 Capital Group (Withdrawn),CCLV,Dominari Securities,3.75,$4.00,$5.00,$16.88,Withdrawn,S/O,Group
14,2026-09-15,OTSAW Ltd. (Withdrawn),OTSA,Aegis Capital,4.40,$4.50,$5.50,$22.00,Withdrawn,S/O,Limited
22,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,$100.00,Withdrawn,S/O,Technologies
28,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,$4.00,$5.00,$9.00,Withdrawn,S/O,Holdings
38,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.00,NaN,NaN,$100.00,Withdrawn,S/O,Others


In [18]:
df_clean['Price Low'] = df['Price Low'].str.replace('$', '', regex=False).astype(float)
df_clean['Price High'] = df['Price High'].str.replace('$', '', regex=False).astype(float)
df_clean['Est $ Vol (millions)'] = df['Est $ Vol (millions)'].str.replace('$', '', regex=False).astype(float)
df_clean['Avg_price'] = df_clean[['Price Low', 'Price High']].mean(axis=1)
df_clean['Shares_offered_value'] = np.where((df_clean['Shares (millions)'] * df_clean['Avg_price']).notna(), df_clean['Shares (millions)'] * df_clean['Avg_price'], df_clean['Est $ Vol (millions)'])
df_clean.info()

<class 'pandas.DataFrame'>
Index: 34 entries, 1 to 499
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   File Date             34 non-null     str    
 1   Company               34 non-null     str    
 2   Symbol                34 non-null     str    
 3   Managers              34 non-null     str    
 4   Shares (millions)     34 non-null     float64
 5   Price Low             28 non-null     float64
 6   Price High            28 non-null     float64
 7   Est $ Vol (millions)  34 non-null     float64
 8   Expected To Trade     34 non-null     str    
 9   SCOOP Rating          34 non-null     str    
 10  Company Type          34 non-null     str    
 11  Avg_price             28 non-null     float64
 12  Shares_offered_value  34 non-null     float64
dtypes: float64(6), str(7)
memory usage: 3.7 KB


In [19]:
print(f"A1. Total withdraw value = {round(df_clean.groupby('Company Type').Shares_offered_value.sum().sort_values(ascending=False).tolist()[0])}")

A1. Total withdraw value = 500


# Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)

In [7]:
url2 = "https://www.iposcoop.com/2025-pricings/"
recent_ipo = pd.read_html(url2)
df2_base = recent_ipo[0]

In [8]:
df2_base['Offer Date'] = pd.to_datetime(df2_base['Offer Date'])
df2 = df2_base[(df2_base['Offer Date'] < '2025-09-01') & (df2_base.Return != 0)]
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 163 entries, 68 to 230
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Company            163 non-null    str           
 1   Symbol             163 non-null    str           
 2   Industry           163 non-null    str           
 3   Offer Date         163 non-null    datetime64[us]
 4   Shares (millions)  163 non-null    float64       
 5   Offer Price        163 non-null    str           
 6   1st Day Close      163 non-null    str           
 7   Current Price      163 non-null    str           
 8   Return             163 non-null    str           
 9   SCOOP Rating       163 non-null    str           
dtypes: datetime64[us](1), float64(1), str(8)
memory usage: 12.9 KB


In [9]:
symbols = df2.Symbol.tolist()
len(symbols)

163

In [10]:
stocks_df = []
stocks_df_missing = []
for s in symbols:
    price = yf.Ticker(s).history(period='2y')[['Open', 'High', 'Low', 'Close', 'Volume']].reset_index()
    if price.empty:
        stocks_df_missing.append(s)
        continue
    price['Date'] = pd.to_datetime(price['Date'])
    price['Symbol'] = s
    price = price[['Date', 'Symbol', 'Open', 'High', 'Low', 'Close', 'Volume']]
    stocks_df.append(price)

stocks_df = pd.concat(stocks_df, ignore_index=True)

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MJID"}}}
$MJID: No data found, symbol may be delisted
$EMPG: No data found, symbol may be delisted
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CAEP"}}}
$CAEP: No data found, symbol may be delisted
$PTNM: No data found, symbol may be delisted
$JENA.U: No data found, symbol may be delisted
$AHL: No data found, symbol may be delisted
$CEPT: No data found, symbol may be delisted
$SDM: No data found, symbol may be delisted
$TBH: No data found, symbol may be delisted
$AGH: No data found, symbol may be delisted
$EPWK: No data found, symbol may be delisted
$MTSR: No data found, symbol may be delisted
$SKBL: No data found, symbol may be delisted
$MCTR: No data found, symbol may be delisted


In [11]:
print(stocks_df_missing)

['MJID', 'EMPG', 'CAEP', 'PTNM', 'JENA.U', 'AHL', 'CEPT', 'SDM', 'TBH', 'AGH', 'EPWK', 'MTSR', 'SKBL', 'MCTR']


In [26]:
stocks_df['growth_252d'] = stocks_df.Close / stocks_df.Close.shift(252) - 1
stocks_df['volatility']  = stocks_df.Close.rolling(30).std() * np.sqrt(252)

risk_free_rate = 0.05
stocks_df['Sharpe'] = (stocks_df['growth_252d'] - risk_free_rate) / stocks_df['volatility']

In [27]:
stocks_df_filter = stocks_df[stocks_df.Date == '2026-09-11']

print(stocks_df_filter.describe().round(4))

           Open      High       Low     Close        Volume  growth_252d  \
count  148.0000  148.0000  148.0000  148.0000  1.480000e+02     148.0000   
mean    12.6963   13.0268   12.3923   12.6630  1.130332e+06       0.0544   
std     19.4271   19.7730   19.0950   19.3515  3.026223e+06       2.8820   
min      0.1160    0.1290    0.1150    0.1250  0.000000e+00      -0.9990   
25%      1.9525    2.1748    1.9475    2.0600  2.060000e+04      -0.8660   
50%      4.3900    4.5700    4.3185    4.4600  1.645000e+05      -0.3870   
75%     17.1175   17.1688   16.1775   16.7325  8.432500e+05       0.0940   
max    120.5000  120.5000  120.4250  120.4250  2.168250e+07      32.6383   

       volatility    Sharpe  
count    148.0000  148.0000  
mean      26.8134       inf  
std       64.4380       NaN  
min        0.0000   -6.0082  
25%        2.2291   -0.1862  
50%        6.9761   -0.0325  
75%       23.8696    0.0017  
max      627.6981       inf  


C:\GitHub\rlp-jym-projects\datatalksclub-zoomcamp\zoomcamp-sma-cohort-2026\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [28]:
print(f"A2: Median Sharpe = {stocks_df_filter.Sharpe.median():.4f}")

A2: Median Sharpe = -0.0325
